# sim-app tutorial

This notebook demonstrates the app-facing `sim_app` workflow:

1. generate one synthetic spatial-transcriptomics dataset,
2. inspect the returned `AnnData`,
3. plot aligned and unaligned coordinates,
4. save the result as `.h5ad`.

It uses small settings so the tutorial stays interactive.

## Setup

From the repository root, create the tutorial environment before opening this notebook:

```bash
bash env/create_tutorial_env.sh
conda activate sim-app-tutorial
jupyter-notebook --no-browser --ip=0.0.0.0 --port 8888
```

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "sim_app").is_dir() and (repo_root.parent / "sim_app").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import sim_app

print("Python:", sys.executable)
print("sim_app module:", getattr(sim_app, "__file__", "<no __file__>"))
print("sim_app version:", getattr(sim_app, "__version__", "<no __version__>"))
print("has generate_data:", hasattr(sim_app, "generate_data"))
print("public names:", [name for name in dir(sim_app) if not name.startswith("_")][:30])

## Generate a small bin-level dataset

`generate_data()` returns exactly one `AnnData` object for the requested output type and slice axis.

In [ ]:
adata = sim_app.generate_data(
    output="bin",
    slice_axis="Z",
    n_cells=250,
    n_slices=3,
    n_domains=4,
    marker_genes_per_type=20,
    sphere_radius_um=200.0,
    capture_window_um=(300.0, 300.0),
    bin_size_um=30.0,
    seed=2025,
)

adata

## Inspect the returned object

`describe()` gives a compact summary of the app-relevant fields.

In [ ]:
summary = sim_app.describe(adata)
summary

In [ ]:
adata.obs.head()

In [ ]:
adata.var.head()

## Plot aligned and unaligned coordinates

`plot()` returns a Matplotlib figure. Use `coordinates="aligned"` for canonical tissue coordinates and `coordinates="unaligned"` for the per-slice rigid transforms.

In [ ]:
fig = sim_app.plot(
    adata,
    view="2d",
    coordinates="aligned",
    color="domain_true",
    point_size=8,
)
fig

In [ ]:
fig = sim_app.plot(
    adata,
    view="2d",
    coordinates="unaligned",
    color="slice_id",
    point_size=8,
)
fig

## Generate other app outputs

Use the same public API for spot-level or cell-level data. Only the requested output is generated.

In [ ]:
spot_adata = sim_app.example_data(output="spot", slice_axis="Y", n_cells=200)
sim_app.describe(spot_adata)

In [ ]:
cell_adata = sim_app.example_data(output="cell", slice_axis="X", n_cells=200)
sim_app.describe(cell_adata)

## Save the result

`save()` writes the object with AnnData's `.h5ad` format and returns the output path.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path = sim_app.save(adata, output_dir / "simulation_bins_z.h5ad")
output_path